## Install ANTLR

In [172]:
# # --- 1. Install Python Runtime ---
# # Install the runtime version that matches the ANTLR tool version
# !pip install antlr4-python3-runtime==4.13.1

# # --- 2. Download ANTLR Tool (Java JAR) ---
# # We'll download version 4.13.1, but any recent version should work
# ANTLR_JAR = "antlr-4.13.1-complete.jar"
# !curl -O https://www.antlr.org/download/{ANTLR_JAR}

# # --- 3. Create a Bash Alias (for easy use of the ANTLR tool) ---
# # We'll save the alias to a temporary file, then source it
# ALIAS_COMMAND = f"alias antlr4='java -jar {ANTLR_JAR}'"
# !echo "{ALIAS_COMMAND}" > ~/.bashrc
# !source ~/.bashrc

# print("Setup complete! ANTLR tool is ready to use.")

In [173]:
!ls -F

__pycache__/               scratch.ipynb
antlr-4.13.1-complete.jar


In [174]:
%%writefile Doggo.g4
grammar Doggo;

// Entry point
start: statement EOF;

// Main statement rules
statement
    : declaration SEMICOLON                     # DeclarationStatement
    | assignment_statement SEMICOLON            # AssignmentStatement
    | expr SEMICOLON                            # ExpressionStatement
    | kennel_definition                         # KennelDefinition
    | import_statement SEMICOLON                # ImportStatement
    | if_statement                              # IfStatement
    | while_statement                           # WhileStatement
    | for_statement                             # ForStatement
    | try_statement                             # TryCatchFinallyStatement
    | raise_statement SEMICOLON                 # GrowlStatement
    | method_call_statement SEMICOLON           # MethodCallStatement
    | list_method_statement SEMICOLON           # ListMethodStatement
    | LEAVE_SCENT '(' expr ',' expr ',' expr ')' SEMICOLON # LeaveScentStatement
    | BURY_BONE '(' expr ')' SEMICOLON          # BuryBoneStatement
    | BEG '(' identifier (',' expr)? ')' SEMICOLON # BegStatement
    | WOOF '(' expr ')' SEMICOLON               # WoofStatement
    ;

// Variable declaration, including list literal assignment
declaration
    : type_ identifier (ASSIGN expr)?           // allow assignment of expr (which can be a list literal)
    ;

// Assignment
assignment_statement : identifier ASSIGN expr;

// Kennel/class definition
kennel_definition : KENNEL ID block;

// Import
import_statement : IMPORT ID;

// Control flow
if_statement
    : IF '(' expr ')' block (SNIFF_ELIF '(' expr ')' block)* (SNIFF_ELSE block)?
    ;

while_statement : WHILE '(' expr ')' block;

for_statement
    : FOR '(' (declaration | assignment_statement | expr)? SEMICOLON expr SEMICOLON (assignment_statement | expr)? ')' block;

// Try-catch-finally
try_statement
    : TRY block sniff_catch_clause* sniff_finally_clause?
    ;

sniff_catch_clause   : SNIFF_CATCH '(' excType=ID excVar=ID ')' block;
sniff_finally_clause : SNIFF_FINALLY block;

// Exceptions
raise_statement : GROWL expr;

// Method calls (object.method(args))
method_call_statement
    : identifier DOT (PEEK | NAP | ID) '(' (expr (',' expr)*)? ')'
    ;

// List methods
list_method_statement
    : identifier DOT LEASHON LPAR element RPAR
    | identifier DOT SNIFFSWAP LPAR INT COMMA element RPAR
    | identifier DOT TUCK LPAR INT COMMA element RPAR
    | identifier DOT UNLEASH LPAR element RPAR
    | identifier DOT SNATCH LPAR RPAR
    | identifier DOT DROPALL LPAR RPAR
    | identifier DOT PAWCOUNT LPAR RPAR
    | identifier DOT HERD LPAR RPAR
    | identifier DOT CLONEPAW LPAR RPAR
    ;

// Blocks
block : LBRACE statement* RBRACE;

// Types (why leash, profile, etc.)
type_
    : TREATS_TYPE
    | WEIGHT_TYPE
    | NAME_TYPE
    | GOODNESS_TYPE
    | DIARY_TYPE
    | LEASH_TYPE     // list
    | PROFILE_TYPE   // dictionary
    | TOYS_TYPE      // set
    | PAWPRINT_TYPE  // tuple
    | PACK_TYPE      // array
    ;

// Expressions (add list_literal as part of expr)
expr
    : atom                        # AtomicExpression
    | expr EXP expr               # Exp
    | expr MUL expr               # MulDiv
    | expr DIV expr               # MulDiv
    | expr FLOOR_DIV expr         # FloorDiv
    | expr MOD expr               # Modulo
    | expr ADD expr               # AddSub
    | expr SUB expr               # AddSub
    | expr EQ expr                # Equal
    | expr NEQ expr               # NotEqual
    | expr GT expr                # GreaterThan
    | expr LT expr                # LessThan
    | expr GTE expr               # GreaterThanEqual
    | expr LTE expr               # LessThanEqual
    | expr IN expr                # In
    | expr NOT IN expr            # NotIn
    | NOT expr                    # Not
    | expr AND expr               # And
    | expr OR expr                # Or
    | list_literal                # ListLiteral
    ;

// List literal definition, e.g., [a, 1, "hi"]
list_literal
    : LBRACK (element (COMMA element)*)? RBRACK
    ;

element
    : INT
    | FLOAT
    | STRING
    | BOOLEAN
    | identifier
    ;

// Atomic expressions
atom
    : INT                              # Number
    | identifier                       # VariableReference
    | STRING                           # StringLiteral
    | EMPTYBOWL                        # NullLiteral
    | '(' expr ')'                     # Parentheses
    | ID '(' (expr (',' expr)*)? ')'   # CallExpression
    | SQRT '(' expr ')'                # SqrtCall
    | PAWSIZE '(' expr ')'             # PawsizeCall
    | SMALLEST_PUP '(' expr (',' expr)+ ')' # SmallestPupCall
    | BIGGEST_PUP '(' expr (',' expr)+ ')'  # BiggestPupCall
    | ALPHA_POWER '(' expr ',' expr ')'     # AlphaPowerCall
    ;

// Identifiers
identifier : ID;

// ===== Lexer rules start here =====

// Operators, delimiters, keywords, etc.
ADD        : '+' ;
SUB        : '-' ;
MUL        : '*' ;
DIV        : '/' ;
FLOOR_DIV  : '//' ;
MOD        : '%' ;
EXP        : '**' ;

EQ         : '==' ;
NEQ        : '!=' ;
GT         : '>' ;
LT         : '<' ;
GTE        : '>=' ;
LTE        : '<=' ;

AND        : 'and' ;
OR         : 'or' ;
NOT        : 'not' ;
IN         : 'in' ;
ASSIGN     : '=' ;
SEMICOLON  : ';' ;
DOT        : '.' ;

LBRACE     : '{' ;
RBRACE     : '}' ;
LBRACK     : '[' ;
RBRACK     : ']' ;
LPAR       : '(' ;
RPAR       : ')' ;
COMMA      : ',' ;

// Control flow
IF         : 'if' ;
SNIFF_ELIF : 'sniff_elif' ;
SNIFF_ELSE : 'sniff_else' ;
WHILE      : 'while' ;
FOR        : 'for' ;

// Try/catch/finally, exceptions
TRY           : 'try' ;
SNIFF_CATCH   : 'sniff_catch' ;
SNIFF_FINALLY : 'sniff_finally' ;
GROWL         : 'growl' ;

// Classes/kennels
KENNEL   : 'kennel' ;
IMPORT   : 'import' ;

// Types
TREATS_TYPE    : 'treats' ;
WEIGHT_TYPE    : 'weight' ;
NAME_TYPE      : 'name' ;
GOODNESS_TYPE  : 'goodness' ;
DIARY_TYPE     : 'diary' ;
LEASH_TYPE     : 'leash' ;
PROFILE_TYPE   : 'profile' ;
TOYS_TYPE      : 'toys' ;
PAWPRINT_TYPE  : 'pawprint' ;
PACK_TYPE      : 'pack' ;

EMPTYBOWL : 'emptybowl' ;

// File methods
PEEK        : 'peek' ;
NAP         : 'nap' ;
LEAVE_SCENT : 'leave_scent' ;
BURY_BONE   : 'bury_bone' ;

// I/O, math, and min/max utilities
BEG          : 'beg' ;
WOOF         : 'woof' ;
SQRT         : 'sqrt' ;
PAWSIZE      : 'pawsize' ;
SMALLEST_PUP : 'smallest_pup' ;
BIGGEST_PUP  : 'biggest_pup' ;
ALPHA_POWER  : 'alpha_power' ;

// List methods
LEASHON   : 'leashon' ;
SNIFFSWAP : 'sniffswap' ;
TUCK      : 'tuck' ;
UNLEASH   : 'unleash' ;
SNATCH    : 'snatch' ;
DROPALL   : 'dropall' ;
PAWCOUNT  : 'pawcount' ;
HERD      : 'herd' ;
CLONEPAW  : 'clonepaw' ;

// Literals and basic types
STRING  : '"' ( ~('\\' | '"') | '\\' . )* '"' ;
INT     : [0-9]+ ;
FLOAT   : [0-9]+ '.' [0-9]+ ;
BOOLEAN : 'true' | 'false' ;
ID      : [a-zA-Z_][a-zA-Z0-9_]* ;

// Whitespace and comments
WS                 : [ \t\r\n]+ -> skip ;
COMMENT_SINGLE_LINE: '#' ~[\r\n]* -> skip ;
COMMENT_MULTI_LINE : '/*' .*? '*/' -> skip ;


Writing Doggo.g4


In [175]:
# Execute the ANTLR tool to generate the Python source files
# -Dlanguage=Python3: Specifies the target language
# -visitor: Generates the DoggoLexerVisitor base class
# The generated files will appear in the Colab file explorer.

!java -jar antlr-4.13.1-complete.jar -Dlanguage=Python3 -visitor Doggo.g4

print("ANTLR Python code generated successfully.")

ANTLR Python code generated successfully.


In [176]:
!ls

Doggo.g4                  DoggoLexer.py             DoggoVisitor.py
Doggo.interp              DoggoLexer.tokens         __pycache__
Doggo.tokens              DoggoListener.py          antlr-4.13.1-complete.jar
DoggoLexer.interp         DoggoParser.py            scratch.ipynb


## Interpreter Class

In [177]:
import sys
sys.path.insert(0, '.') # Add current directory to Python path

import importlib
from antlr4 import *
from antlr4.error.ErrorListener import ErrorListener # Import the ErrorListener class
from antlr4.tree.Tree import TerminalNode # Import TerminalNode
import os # Import the os module for file operations
import math # Import math module for sqrt, pow

# Explicitly reload the modules
import DoggoLexer
import DoggoParser
import DoggoVisitor
importlib.reload(DoggoLexer)
importlib.reload(DoggoParser)
importlib.reload(DoggoVisitor)


from DoggoLexer import DoggoLexer
from DoggoParser import DoggoParser
from DoggoVisitor import DoggoVisitor # This is the base class, we'll use our custom interpreter below


# Custom Error Listener to suppress default error messages
class SuppressErrorListener(ErrorListener):
    def syntaxError(self, recognizer, offendingSymbol, line, column, msg, e):
        pass

    def reportAmbiguity(self, recognizer, dfa, startIndex, stopIndex, exact, ambigAlts, configs):
        pass

    def reportAttemptingFullContext(self, recognizer, dfa, startIndex, stopIndex, conflictingAlts, configs):
        pass

    def reportContextSensitivity(self, recognizer, dfa, startIndex, stopIndex, prediction, configs):
        pass

# Helper function for file extension validation
def validate_file_extension(filename):
    allowed_extensions = ('.csv', '.txt')
    if not filename.endswith(allowed_extensions):
        raise ValueError(f"Unsupported file extension for '{filename}'. Only {', '.join(allowed_extensions)} are supported.")

# Override the Visitor methods to perform the calculations
class DoggoInterpreter(DoggoVisitor):
    def __init__(self, variables=None):
        # Initialize a dictionary to store variables, or use the provided one
        self.variables = variables if variables is not None else {}
        self.file_handles = {}

    # Visit a parse tree produced by DoggoParser#start.
    def visitStart(self, ctx:DoggoParser.StartContext):
        return self.visit(ctx.statement())

    # Visit a parse tree produced by DoggoParser#DeclarationStatement.
    def visitDeclarationStatement(self, ctx:DoggoParser.DeclarationStatementContext):
        # A DeclarationStatement wraps a declaration rule
        return self.visit(ctx.declaration())

    # Visit a parse tree produced by DoggoParser#AssignmentStatement.
    def visitAssignmentStatement(self, ctx:DoggoParser.AssignmentStatementContext):
        # An AssignmentStatement wraps an assignment_statement rule
        return self.visit(ctx.assignment_statement())

    # Visit a parse tree produced by DoggoParser#ExpressionStatement.
    def visitExpressionStatement(self, ctx:DoggoParser.ExpressionStatementContext):
        return self.visit(ctx.expr())

    # Visit a parse tree produced by DoggoParser#declaration.
    # def visitDeclaration(self, ctx:DoggoParser.DeclarationContext):
    #     variable_name = ctx.identifier().getText()
    #     # We are not strictly enforcing types in this simple interpreter,
    #     # but we can store the declared type if needed for later type checking.
    #     declared_type = ctx.type_().getText()

    #     # Check if there's an assignment
    #     if ctx.expr():
    #         value = self.visit(ctx.expr())
    #         # Assign value directly without float() conversion to allow None
    #         self.variables[variable_name] = value
    #     else:
    #         # Declare variable without assignment (initialize to None or a default value)
    #         self.variables[variable_name] = None

    #     return None # Declarations don't return a value in this context
    
    
    def visitDeclaration(self, ctx):
        variable_name = ctx.identifier().getText()
        declared_type = ctx.type_().getText()
        if ctx.expr():
            value = self.visit(ctx.expr())  # <--- This should pick up the actual RHS!
        else:
            value = [] if declared_type == "leash" else None
        self.variables[variable_name] = value
        return None


    # Visit a parse tree produced by DoggoParser#assignment_statement.
    def visitAssignment_statement(self, ctx:DoggoParser.Assignment_statementContext):
        variable_name = ctx.identifier().getText()
        value = self.visit(ctx.expr())
        # Assign value directly without float() conversion to allow None
        self.variables[variable_name] = value
        return None

    # Visit a parse tree produced by DoggoParser#KennelDefinition.
    def visitKennelDefinition(self, ctx:DoggoParser.KennelDefinitionContext):
        # ctx here is the context for the labeled alternative '#KennelDefinition' within 'statement'.
        # It contains the actual 'kennel_definition' rule context as a child,
        # which can be accessed via ctx.kennel_definition() (common ANTLR pattern for labeled alternatives).

        # Get the actual kennel_definition rule context
        # (kennel_definition : KENNEL ID block;)
        kennel_rule_ctx = ctx.kennel_definition() # Assuming this method exists on the generated context

        if kennel_rule_ctx is None:
            raise ValueError("Could not find the 'kennel_definition' rule context within the labeled alternative.")

        # Now, from kennel_rule_ctx, we can directly access its components
        # KENNEL is token, ID is token, block is rule
        # ANTLR usually generates methods like ID() for direct token references in rules.
        kennel_name_token = kennel_rule_ctx.ID()

        if kennel_name_token is None:
            raise ValueError("No ID token found in the 'kennel_definition' rule context.")

        kennel_name = kennel_name_token.getText()

        print(f"DEBUG: Successfully extracted Kennel Name: {kennel_name}")

        block_ctx = kennel_rule_ctx.block() # Access the block context directly from the actual rule context

        if block_ctx:
            print(f"DEBUG: Processing block for kennel '{kennel_name}'.")
            self.visit(block_ctx)
        else:
            print(f"DEBUG: No block found for kennel '{kennel_name}'.")
        return None

    # Visit a parse tree produced by DoggoParser#ImportStatement.
    def visitImportStatement(self, ctx:DoggoParser.ImportStatementContext):
        print(f"DEBUG: Entering visitImportStatement for context type: {type(ctx)}")
        # The DoggoParser.ImportStatementContext is the specific context for the #ImportStatement labeled alternative.
        # It typically contains a method named after the rule it wraps, in this case 'import_statement'.
        import_stmt_rule_ctx = ctx.import_statement()

        if import_stmt_rule_ctx:
            print(f"DEBUG: Retrieved import_statement rule context type: {type(import_stmt_rule_ctx)}")
            # The import_statement rule context should have an ID method for the 'ID' token.
            id_token = import_stmt_rule_ctx.ID()
            if id_token:
                module_name = id_token.getText()
                print(f"DEBUG: Importing module: {module_name}")
            else:
                print(f"DEBUG: No ID token found in import_statement rule context.")
        else:
            print(f"DEBUG: import_statement rule context not found within ImportStatementContext.")
        return None

    # Visit a parse tree produced by DoggoParser#IfStatement.
    def visitIfStatement(self, ctx:DoggoParser.IfStatementContext):
        print(f"DEBUG: Entering visitIfStatement for context type: {type(ctx)}")

        # Get the actual if_statement rule context from the labeled alternative context
        if_stmt_rule_ctx = ctx.if_statement()

        if if_stmt_rule_ctx is None:
            raise ValueError("Could not find the 'if_statement' rule context within the labeled alternative IfStatementContext.")

        # Evaluate the 'if' condition (first expr child of the if_stmt_rule_ctx)
        if_condition_result = self.visit(if_stmt_rule_ctx.expr(0))
        if if_condition_result:
            self.visit(if_stmt_rule_ctx.block(0))
            return None

        # Evaluate 'sniff_elif' conditions
        # Loop through the SNIFF_ELIF tokens. Each corresponds to an expr and a block.
        # The first expr and block are for the 'if', so 'elif' conditions start from index 1 for expr and block.
        for i in range(len(if_stmt_rule_ctx.SNIFF_ELIF())):
            # The expr for the i-th sniff_elif is at index i+1
            elif_condition_result = self.visit(if_stmt_rule_ctx.expr(i + 1))
            if elif_condition_result:
                # The block for the i-th sniff_elif is at index i+1
                self.visit(if_stmt_rule_ctx.block(i + 1))
                return None

        # Evaluate 'sniff_else' block if present and no other condition was true
        if if_stmt_rule_ctx.SNIFF_ELSE():
            # The 'else' block, if it exists, is always the last block in the list of blocks from if_stmt_rule_ctx
            self.visit(if_stmt_rule_ctx.block(len(if_stmt_rule_ctx.block()) - 1))
        return None

    # Visit a parse tree produced by DoggoParser#WhileStatement.
    def visitWhileStatement(self, ctx:DoggoParser.WhileStatementContext):
        print(f"DEBUG: Entering visitWhileStatement for context type: {type(ctx)}")
        while_stmt_rule_ctx = ctx.while_statement()
        if while_stmt_rule_ctx is None:
            raise ValueError("Could not find the 'while_statement' rule context within the labeled alternative WhileStatementContext.")

        condition_expr = while_stmt_rule_ctx.expr()
        loop_block = while_stmt_rule_ctx.block()

        iteration = 0
        while self.visit(condition_expr):
            iteration += 1
            print(f"DEBUG: While loop iteration {iteration}. Condition: {self.visit(condition_expr)}. x={self.variables.get('x')}, y={self.variables.get('y')}")
            self.visit(loop_block)
            print(f"DEBUG: After while block execution. x={self.variables.get('x')}, y={self.variables.get('y')}")
            if iteration > 100: # Add a safeguard for infinite loops
                print("WARNING: While loop exceeded 100 iterations. Breaking to prevent infinite loop.")
                break
        print(f"DEBUG: While loop condition evaluated to False. Exiting loop.")
        return None

    # Visit a parse tree produced by DoggoParser#ForStatement.
    def visitForStatement(self, ctx:DoggoParser.ForStatementContext):
        print(f"DEBUG: Entering visitForStatement for context type: {type(ctx)}")
        for_stmt_rule_ctx = ctx.for_statement()
        if for_stmt_rule_ctx is None:
            raise ValueError("Could not find the 'for_statement' rule context within the labeled alternative ForStatementContext.")

        init_part_ctx = None
        condition_expr_ctx = None
        increment_part_ctx = None
        loop_block_ctx = for_stmt_rule_ctx.block()

        # Extract init part: (declaration | assignment_statement | expr)?
        # The grammar has 'for ( (declaration | assignment_statement | expr)? SEMICOLON expr SEMICOLON (assignment_statement | expr)? ) block'
        # The first child after '(' (index 2) could be the init part or the first SEMICOLON.

        children_in_paren = []
        in_paren = False
        for child in for_stmt_rule_ctx.children:
            if isinstance(child, TerminalNode) and child.getSymbol().text == '(':
                in_paren = True
                continue
            if isinstance(child, TerminalNode) and child.getSymbol().text == ')':
                in_paren = False
                break
            if in_paren:
                children_in_paren.append(child)

        semicolon_indices_in_paren = [i for i, child in enumerate(children_in_paren) if isinstance(child, TerminalNode) and child.getSymbol().type == DoggoLexer.SEMICOLON]

        if len(semicolon_indices_in_paren) != 2:
            raise ValueError("For loop structure invalid: expected exactly two semicolons inside parentheses.")

        # Extract init part
        if semicolon_indices_in_paren[0] > 0: # If there's anything before the first semicolon
            # The init part is the single child before the first semicolon
            init_part_ctx = children_in_paren[0]

        # Extract condition part (between first and second semicolon)
        # Iterate children between the semicolons to find the expr
        for i in range(semicolon_indices_in_paren[0] + 1, semicolon_indices_in_paren[1]):
            child = children_in_paren[i]
            if isinstance(child, DoggoParser.ExprContext):
                condition_expr_ctx = child
                break

        if condition_expr_ctx is None:
            raise ValueError("For loop condition expression is missing or malformed.")

        # Extract increment part (after second semicolon)
        if semicolon_indices_in_paren[1] < len(children_in_paren) -1: # If there's anything after the second semicolon
            # The increment part is the single child after the second semicolon
            increment_part_ctx = children_in_paren[semicolon_indices_in_paren[1] + 1]

        print(f"DEBUG: For loop init part context: {type(init_part_ctx).__name__ if init_part_ctx else 'None'}")
        print(f"DEBUG: For loop condition expression context: {type(condition_expr_ctx).__name__}")
        print(f"DEBUG: For loop increment part context: {type(increment_part_ctx).__name__ if increment_part_ctx else 'None'}")

        # The loop block is the last child of the for_statement rule.
        loop_block = for_stmt_rule_ctx.block()
        if loop_block is None:
            raise ValueError("For loop body (block) is missing.")
        print(f"DEBUG: For loop block context: {type(loop_block).__name__}")

        # Execute initialization part
        if init_part_ctx:
            self.visit(init_part_ctx)

        # Loop execution
        iteration = 0
        while self.visit(condition_expr_ctx):
            iteration += 1
            print(f"DEBUG: For loop iteration {iteration}. Condition: {self.visit(condition_expr_ctx)}. i={self.variables.get('i')}, sum_for={self.variables.get('sum_for')}")
            self.visit(loop_block)
            if increment_part_ctx:
                self.visit(increment_part_ctx)
            print(f"DEBUG: After for block and increment. i={self.variables.get('i')}, sum_for={self.variables.get('sum_for')}")
            if iteration > 100: # Add a safeguard for infinite loops
                print("WARNING: For loop exceeded 100 iterations. Breaking to prevent infinite loop.")
                break
        print(f"DEBUG: For loop condition evaluated to False. Exiting loop.")
        return None

    # New: Visit a parse tree produced by DoggoParser#TryCatchFinallyStatement.
    def visitTryCatchFinallyStatement(self, ctx:DoggoParser.TryCatchFinallyStatementContext):
        print(f"DEBUG: Entering visitTryCatchFinallyStatement for context type: {type(ctx)}")
        try_stmt_rule_ctx = ctx.try_statement()

        try:
            self.visit(try_stmt_rule_ctx.block()) # Visit the 'try' block
        except Exception as e:
            caught = False
            # Visit 'sniff_catch' clauses
            for i in range(len(try_stmt_rule_ctx.sniff_catch_clause())):
                catch_clause_ctx = try_stmt_rule_ctx.sniff_catch_clause(i)
                # In a real interpreter, you'd check if 'e' matches the type of exception caught by this clause.
                # For now, we'll just execute the first sniff_catch block that exists.
                print(f"DEBUG: Catching exception: {e}")
                # Now we extract the variable name from the catch clause
                exception_variable_name = catch_clause_ctx.excVar.text # Changed getText() to .text
                # Assign the exception object (e) to the variable name in the current scope
                self.variables[exception_variable_name] = e
                self.visit(catch_clause_ctx.block()) # Visit the 'sniff_catch' block
                caught = True
                break # Only enter one catch block
            if not caught:
                raise # Re-raise if not caught by any sniff_catch block
        finally:
            # Visit 'sniff_finally' clause if present
            if try_stmt_rule_ctx.sniff_finally_clause():
                print("DEBUG: Executing sniff_finally block.")
                self.visit(try_stmt_rule_ctx.sniff_finally_clause().block())
        return None

    # New: Visit a parse tree produced by DoggoParser#GrowlStatement.
    def visitGrowlStatement(self, ctx:DoggoParser.GrowlStatementContext):
        print(f"DEBUG: Entering visitGrowlStatement for context type: {type(ctx)}")
        raise_stmt_rule_ctx = ctx.raise_statement()
        error_message = self.visit(raise_stmt_rule_ctx.expr())
        print(f"DEBUG: Raising exception with message: {error_message}")
        raise Exception(error_message)

    # New: Visit a parse tree produced by DoggoParser#StringLiteral.
    def visitStringLiteral(self, ctx:DoggoParser.StringLiteralContext):
        # Remove the quotes from the string literal
        return ctx.STRING().getText()[1:-1]

    # New: Visit a parse tree produced by DoggoParser#NullLiteral.
    def visitNullLiteral(self, ctx:DoggoParser.NullLiteralContext):
        return None

    # New: Visit a parse tree produced by DoggoParser#CallExpression.
    def visitCallExpression(self, ctx:DoggoParser.CallExpressionContext):
        func_name = ctx.ID().getText()
        args = [self.visit(e) for e in ctx.expr()] # expr() returns a list of expr contexts

        if func_name == "Exception" and len(args) == 1:
            # Special handling for Exception("...") to extract the message
            return args[0] # Return the message string
        else:
            raise NotImplementedError(f"Function call '{func_name}' not implemented yet, or invalid arguments for Exception.")

    # New: Visit a parse tree produced by DoggoParser#MethodCallStatement.
    def visitMethodCallStatement(self, ctx:DoggoParser.MethodCallStatementContext):
        # The grammar rule is: method_call_statement : identifier DOT (PEEK | NAP | ID) '(' (expr (',' expr)*)? ')' ;
        # The 'ctx' passed here is for the labeled alternative '#MethodCallStatement'.
        # We need to get the actual 'method_call_statement' rule context from it.
        method_call_stmt_rule_ctx = ctx.method_call_statement()

        if method_call_stmt_rule_ctx is None:
            raise ValueError("Could not find the 'method_call_statement' rule context within the labeled alternative.")

        # Now, from the actual rule context, we can get its components.
        # target_object_name is from the 'identifier' rule, which itself contains an 'ID' token.
        target_object_name = method_call_stmt_rule_ctx.identifier().ID().getText()
        # method_name is directly obtained from the specific method token (PEEK, NAP, or generic ID)
        method_token = method_call_stmt_rule_ctx.getChild(2) # The 3rd child is the method name token
        method_name = method_token.getText()

        args = [self.visit(e) for e in method_call_stmt_rule_ctx.expr()] # expr() returns a list of expr contexts for arguments

        if target_object_name not in self.variables:
            raise NameError(f"Object '{target_object_name}' not found.")

        target_object = self.variables[target_object_name]

        if method_name == "peek":
            if not args or not isinstance(args[0], str):
                raise ValueError("peek method requires a string filename as an argument.")
            filename = args[0]
            validate_file_extension(filename) # Validate file extension
            try:
                with open(filename, 'r') as f:
                    content = f.read()
                self.variables[target_object_name] = content
                print(f"DEBUG: Successfully read content from '{filename}' into '{target_object_name}'.")
            except FileNotFoundError:
                raise FileNotFoundError(f"File '{filename}' not found for peek operation.")
            except Exception as e:
                raise RuntimeError(f"Error peeking file '{filename}': {e}")
        elif method_name == "nap":
            if args:
                raise ValueError("nap method does not take any arguments.")
            if target_object_name in self.variables:
                self.variables[target_object_name] = "" # Simulate closing by clearing content
                print(f"DEBUG: Successfully 'napped' (closed/cleared) '{target_object_name}'.")
            else:
                print(f"DEBUG: Attempted to nap non-existent diary '{target_object_name}'.")
        else:
            raise NotImplementedError(f"Method '{method_name}' not implemented for object '{target_object_name}'.")
        return None

    # New: Visit a parse tree produced by DoggoParser#LeaveScentStatement.
    def visitLeaveScentStatement(self, ctx:DoggoParser.LeaveScentStatementContext):
        # The grammar rule is: LEAVE_SCENT '(' expr ',' expr ',' expr ')' SEMICOLON # LeaveScentStatement

        # Access the arguments directly from the expr children of the LeaveScentStatementContext
        file_op = self.visit(ctx.expr(0))
        filename = self.visit(ctx.expr(1))
        content_to_write = self.visit(ctx.expr(2))

        if not all(isinstance(arg, str) for arg in [file_op, filename, content_to_write]):
            raise ValueError("leave_scent function requires 3 string arguments: file_op, filename, content.")

        validate_file_extension(filename) # Validate file extension

        if file_op not in ['a', 'w']:
            raise ValueError(f"Invalid file operation mode '{file_op}'. Must be 'a' (append) or 'w' (write).")

        try:
            with open(filename, file_op) as f:
                f.write(content_to_write)
            print(f"DEBUG: Successfully '{file_op}' content to '{filename}' using leave_scent.")
        except IOError as e:
            raise IOError(f"Error writing to file '{filename}' with mode '{file_op}': {e}")
        except Exception as e:
            raise RuntimeError(f"An unexpected error occurred during leave_scent operation on '{filename}': {e}")
        return None

    # New: Visit a parse tree produced by DoggoParser#BuryBoneStatement.
    def visitBuryBoneStatement(self, ctx:DoggoParser.BuryBoneStatementContext):
        # The grammar rule is: BURY_BONE '(' expr ')' SEMICOLON # BuryBoneStatement

        # Access the filename argument from the expr child of the BuryBoneStatementContext
        filename = self.visit(ctx.expr())

        if not isinstance(filename, str):
            raise ValueError("bury_bone function requires a string filename as an argument.")

        validate_file_extension(filename) # Validate file extension

        try:
            os.remove(filename)
            print(f"DEBUG: Successfully deleted file: '{filename}'.")
        except FileNotFoundError:
            print(f"DEBUG: Attempted to delete non-existent file: '{filename}'.")
            # Optionally, you could raise an error here if deleting non-existent files should be an error.
            # raise FileNotFoundError(f"File '{filename}' not found for bury_bone operation.")
        except OSError as e:
            raise OSError(f"Error deleting file '{filename}': {e}")
        except Exception as e:
            raise RuntimeError(f"An unexpected error occurred during bury_bone operation on '{filename}': {e}")
        return None

    # New: Visit a parse tree produced by DoggoParser#BegStatement.
    def visitBegStatement(self, ctx:DoggoParser.BegStatementContext):
        # The grammar rule is: BEG '(' identifier (',' expr)? ')' SEMICOLON # BegStatement
        variable_name = ctx.identifier().getText()
        prompt = ""
        # Get the single optional 'expr' context. For BEG, there will be 0 or 1.
        prompt_expression = ctx.expr() # Call expr() without arguments to get the single context or None
        if prompt_expression: # Check if the expression for the prompt is present
            # If there's an expression for the prompt, evaluate it.
            prompt = str(self.visit(prompt_expression))

        # In a Colab environment, 'input()' can be tricky with non-blocking execution.
        # For now, we'll simulate input or return a default value.
        # If running in a truly interactive Python interpreter, input() would work directly.
        try:
            user_input = input(prompt) # This will block execution in Colab
            self.variables[variable_name] = user_input
            print(f"DEBUG: Begged input '{user_input}' into '{variable_name}'.")
        except Exception as e:
            # Fallback for non-interactive environments or errors
            print(f"WARNING: Could not 'beg' for input (possibly non-interactive environment). Assigning default value 'fallback_input' to '{variable_name}'. Error: {e}")
            self.variables[variable_name] = "fallback_input" # Assign a default or handle as error
        return None

    # New: Visit a parse tree produced by DoggoParser#WoofStatement.
    def visitWoofStatement(self, ctx:DoggoParser.WoofStatementContext):
        # The grammar rule is: WOOF '(' expr ')' SEMICOLON # WoofStatement
        value_to_print = self.visit(ctx.expr())
        print(value_to_print)
        print(f"DEBUG: Woofed: '{value_to_print}'")
        return None

    # New: Visit a parse tree produced by DoggoParser#SqrtCall.
    def visitSqrtCall(self, ctx:DoggoParser.SqrtCallContext):
        value = self.visit(ctx.expr())
        if not isinstance(value, (int, float)) or value < 0:
            raise ValueError(f"sqrt() expects a non-negative number, got {value}")
        return math.sqrt(value)

    # New: Visit a parse tree produced by DoggoParser#PawsizeCall (for abs).
    def visitPawsizeCall(self, ctx:DoggoParser.PawsizeCallContext):
        value = self.visit(ctx.expr())
        if not isinstance(value, (int, float)):
            raise ValueError(f"pawsize() expects a number, got {value}")
        return abs(value)

    # New: Visit a parse tree produced by DoggoParser#SmallestPupCall (for min).
    def visitSmallestPupCall(self, ctx:DoggoParser.SmallestPupCallContext):
        values = [self.visit(e) for e in ctx.expr()]
        if not all(isinstance(v, (int, float)) for v in values):
            raise ValueError(f"smallest_pup() expects numbers, got {values}")
        return min(values)

    # New: Visit a parse tree produced by DoggoParser#BiggestPupCall (for max).
    def visitBiggestPupCall(self, ctx:DoggoParser.BiggestPupCallContext):
        values = [self.visit(e) for e in ctx.expr()]
        if not all(isinstance(v, (int, float)) for v in values):
            raise ValueError(f"biggest_pup() expects numbers, got {values}")
        return max(values)

    # New: Visit a parse tree produced by DoggoParser#AlphaPowerCall (for pow).
    def visitAlphaPowerCall(self, ctx:DoggoParser.AlphaPowerCallContext):
        base = self.visit(ctx.expr(0))
        exponent = self.visit(ctx.expr(1))
        if not all(isinstance(v, (int, float)) for v in [base, exponent]):
            raise ValueError(f"alpha_power() expects numbers, got base={base}, exponent={exponent}")
        return math.pow(base, exponent)

    # Visit a parse tree produced by DoggoParser#block.
    def visitBlock(self, ctx:DoggoParser.BlockContext):
        for statement_ctx in ctx.statement():
            self.visit(statement_ctx)
        return None

    # Visit a parse tree produced by DoggoParser#MulDiv.
    def visitMulDiv(self, ctx:DoggoParser.MulDivContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        # Check the token type for multiplication or division
        op_token_type = ctx.getChild(1).getSymbol().type
        if op_token_type == DoggoLexer.MUL:
            return left * right
        elif op_token_type == DoggoLexer.DIV:
            # Handle division by zero if necessary
            if right == 0:
                raise ValueError("Division by zero")
            return left / right

    # Visit a parse tree produced by DoggoParser#AddSub.
    def visitAddSub(self, ctx:DoggoParser.AddSubContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        # Check the token type for addition or subtraction
        op_token_type = ctx.getChild(1).getSymbol().type
        if op_token_type == DoggoLexer.ADD:
            return left + right
        elif op_token_type == DoggoLexer.SUB:
            return left - right

    # Visit a parse tree produced by DoggoParser#FloorDiv.
    def visitFloorDiv(self, ctx:DoggoParser.FloorDivContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        if right == 0:
             raise ValueError("Division by zero")
        return left // right

    # Visit a parse tree produced by DoggoParser#Modulo.
    def visitModulo(self, ctx:DoggoParser.ModuloContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        if right == 0:
             raise ValueError("Modulo by zero")
        return left % right

    # Visit a parse tree produced by DoggoParser#Exp.
    def visitExp(self, ctx:DoggoParser.ExpContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left ** right

    # Visit a parse tree produced by DoggoParser#Equal.
    def visitEqual(self, ctx:DoggoParser.EqualContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left == right

    # Visit a parse tree produced by DoggoParser#NotEqual.
    def visitNotEqual(self, ctx:DoggoParser.NotEqualContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left != right

    # Visit a parse tree produced by DoggoParser#GreaterThan.
    def visitGreaterThan(self, ctx:DoggoParser.GreaterThanContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left > right

    # Visit a parse tree produced by DoggoParser#LessThan.
    def visitLessThan(self, ctx:DoggoParser.LessThanContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left < right

    # Visit a parse tree produced by DoggoParser#GreaterThanEqual.
    def visitGreaterThanEqual(self, ctx:DoggoParser.GreaterThanEqualContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left >= right

    # Visit a parse tree produced by DoggoParser#LessThanEqual.
    def visitLessThanEqual(self, ctx:DoggoParser.LessThanEqualContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left <= right

    # Visit a parse tree produced by DoggoParser#And.
    def visitAnd(self, ctx:DoggoParser.AndContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left and right

    # Visit a parse tree produced by DoggoParser#Or.
    def visitOr(self, ctx:DoggoParser.OrContext):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        return left or right

    # Visit a parse tree produced by DoggoParser#Not.
    def visitNot(self, ctx:DoggoParser.NotContext):
        return not self.visit(ctx.expr())

    # Visit a parse tree produced by DoggoParser#In.
    def visitIn(self, ctx):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        # Assuming 'in' is used for checking if an element is in a collection
        # You would need to extend the grammar and interpreter to handle collections (lists, tuples, etc.)
        # For now, this will likely result in a TypeError if the right side is not a collection.
        return left in right


    # Visit a parse tree produced by DoggoParser#NotIn.
    def visitNotIn(self, ctx):
        left = self.visit(ctx.expr(0))
        right = self.visit(ctx.expr(1))
        # Similar to visitIn, this requires handling collections.
        return left not in right


    # Visit a parse tree produced by DoggoParser#Number..
    def visitNumber(self, ctx:DoggoParser.NumberContext):
        return float(ctx.INT().getText())

    # Visit a parse tree produced by DoggoParser#VariableReference.
    def visitVariableReference(self, ctx:DoggoParser.VariableReferenceContext):
        variable_name = ctx.identifier().getText()
        if variable_name in self.variables:
            return self.variables[variable_name]
        else:
            raise NameError(f"Variable '{variable_name}' is not defined.")

    # Visit a parse tree produced by DoggoParser#Parentheses.
    def visitParentheses(self, ctx:DoggoParser.ParenthesesContext):
        return self.visit(ctx.expr()) # Just return the evaluation of the expression inside the parentheses

    # Add visits for the type rules - these simply return the text of the type
    def visitType_(self, ctx:DoggoParser.Type_Context):
        return ctx.getText()
    
    
    def visitListLiteral(self, ctx):
        for child in ctx.children:
            txt = child.getText()
            print("DEBUG: ListLiteral text child:", txt)
            # If the child is one quoted list string: ["Max","Rex"]
            # Remove [ and ], split on comma, strip quotes
            if txt.startswith("[") and txt.endswith("]"):
                contents = txt[1:-1]
                if contents:
                    items = [item.strip().strip('"') for item in contents.split(",")]
                    print("DEBUG: List literal fallback parsed items:", items)
                    return items
        print("DEBUG: ListLiteral unexpected child structure")
        return []


    def visitListMethodStatement(self, ctx):
        # If the first child is a List_method_statementContext, go one level deeper
        first_child = ctx.children[0]
        if isinstance(first_child, DoggoParser.List_method_statementContext):
            # Recursively call this function with the actual content context
            return self.visitListMethodStatement(first_child)

        # Proceed as normal: find identifier, method, and arguments
        varname = None
        for child in ctx.children:
            if isinstance(child, DoggoParser.IdentifierContext):
                varname = child.getText()
                break
            if isinstance(child, TerminalNode) and child.symbol.type == DoggoLexer.ID:
                varname = child.getText()
                break
        if varname is None:
            print("DEBUG: ListMethodStatement child types:", [type(c) for c in ctx.children])
            print("DEBUG: ListMethodStatement texts:", [c.getText() for c in ctx.children])
            raise RuntimeError("Could not find leash variable name in list method statement.")

        if varname not in self.variables or not isinstance(self.variables[varname], list):
            raise TypeError(f"Variable '{varname}' must be a leash (list)")
        the_list = self.variables[varname]

        method = None
        args = []
        for child in ctx.children:
            if isinstance(child, TerminalNode) and child.getText() in {
                "leashon", "sniffswap", "tuck", "unleash", "snatch", "dropall", "pawcount", "herd", "clonepaw"}:
                method = child.getText()
            if isinstance(child, DoggoParser.ElementContext):
                args.append(self.visit(child))
            if isinstance(child, TerminalNode) and child.symbol.type == DoggoLexer.INT:
                args.append(int(child.getText()))

        if method == "leashon":
            if args:
                the_list.append(args[-1])
        elif method == "sniffswap":
            if len(args) >= 2:
                the_list[args[0]] = args[1]
        elif method == "tuck":
            if len(args) >= 2:
                the_list.insert(args[0], args[1])
        elif method == "unleash":
            if args:
                the_list.remove(args[-1])
        elif method == "snatch":
            if the_list:
                the_list.pop()
        elif method == "dropall":
            the_list.clear()
        elif method == "pawcount":
            return len(the_list)
        elif method == "herd":
            the_list.sort()
        elif method == "clonepaw":
            return list(the_list)
        return None


    def visitElement(self, ctx):
        # Print for debugging
        print("DEBUG: Element context type:", type(ctx))
        print("DEBUG: Element children:", [c.getText() for c in ctx.children])
        for child in ctx.getChildren():
            # If child is a TerminalNode for STRING
            if isinstance(child, TerminalNode) and child.symbol.type == DoggoLexer.STRING:
                val = child.getText().strip('"')
                print(f"DEBUG: visitElement fallback STRING = {val}")
                return val
            elif isinstance(child, TerminalNode) and child.symbol.type == DoggoLexer.INT:
                val = int(child.getText())
                print(f"DEBUG: visitElement fallback INT = {val}")
                return val
            elif isinstance(child, TerminalNode) and child.symbol.type == DoggoLexer.FLOAT:
                val = float(child.getText())
                print(f"DEBUG: visitElement fallback FLOAT = {val}")
                return val
            # Add similar branches for other types if needed
        print("DEBUG: visitElement fallback None case")
        return None


## Test Case

In [178]:
def run_interpreter(input_string, variables=None):
    # 1. Create a stream of characters from the input
    input_stream = InputStream(input_string)

    # 2. Create the Lexer (tokenizer)
    lexer = DoggoLexer(input_stream)
    # Remove default error listeners and add our custom one
    lexer.removeErrorListeners()
    lexer.addErrorListener(SuppressErrorListener())


    # 3. Create a stream of tokens
    token_stream = CommonTokenStream(lexer)

    # 4. Create the Parser
    parser = DoggoParser(token_stream)
    # Remove default error listeners and add our custom one
    parser.removeErrorListeners()
    parser.addErrorListener(SuppressErrorListener())


    # 5. Get the root of the Parse Tree by calling the starting rule
    tree = parser.start()

    # Print the parse tree
    print("-- Parse Tree --")
    print(tree.toStringTree(recog=parser))
    print("------------------")

    # 6. Create our custom Interpreter (Visitor) and start the traversal (interpretation)
    interpreter = DoggoInterpreter(variables) # Pass the variables dictionary
    result = interpreter.visit(tree)

    return result, interpreter.variables # Return both the result and the updated variables

# Initialize variables dictionary
my_variables = {}

# Test case for variable declaration with assignment
input_declaration_1 = "treats my_treats = 10;"
print(f"\nInput: {input_declaration_1}")
_, my_variables = run_interpreter(input_declaration_1, my_variables)

# Test case for variable declaration without assignment
input_declaration_2 = "weight my_weight;"
print(f"\nInput: {input_declaration_2}")
_, my_variables = run_interpreter(input_declaration_2, my_variables)

# Test using a declared variable in an expression
input_expression_1 = "my_treats + 5;" # Should evaluate to 15.0
print(f"\nInput: {input_expression_1}")
output_1_expr, my_variables = run_interpreter(input_expression_1, my_variables)
if output_1_expr is not None: print(f"Result: {output_1_expr}")


##############TEST FOR LIST
# Test case for leash (list) declaration with initialization
input_leash_decl = 'leash dog_names = ["Max", "Rex"];'
print(f"\nInput: {input_leash_decl}")
_, my_variables = run_interpreter(input_leash_decl, my_variables)
print("Variables after leash declaration:", my_variables)

# Test case for leash method: append/leashon
input_leash_append = 'dog_names.leashon("Buddy");'
print(f"\nInput: {input_leash_append}")
_, my_variables = run_interpreter(input_leash_append, my_variables)
print("Variables after leashon:", my_variables)

# Test case for leash method: length/pawcount
input_leash_len = 'dog_names.pawcount();'
print(f"\nInput: {input_leash_len}")
len_result, my_variables = run_interpreter(input_leash_len, my_variables)
print(f"Result: {len_result}")

# Test case for leash method: remove/unleash
input_leash_remove = 'dog_names.unleash("Rex");'
print(f"\nInput: {input_leash_remove}")
_, my_variables = run_interpreter(input_leash_remove, my_variables)
print("Variables after unleash:", my_variables)

# Test case for leash method: pop/snatch
input_leash_pop = 'dog_names.snatch();'
print(f"\nInput: {input_leash_pop}")
_, my_variables = run_interpreter(input_leash_pop, my_variables)
print("Variables after snatch:", my_variables)
##############END TEST FOR LIST



# Test using another arithmetic operation with a variable
input_expression_2 = "my_treats * 2;" # Should evaluate to 20.0
print(f"\nInput: {input_expression_2}")
output_2_expr, my_variables = run_interpreter(input_expression_2, my_variables)
if output_2_expr is not None: print(f"Result: {output_2_expr}")

# Test a comparison with a variable
input_expression_3 = "my_treats > 5;" # Should evaluate to True
print(f"\nInput: {input_expression_3}")
output_3_expr, my_variables = run_interpreter(input_expression_3, my_variables)
if output_3_expr is not None: print(f"Result: {output_3_expr}")

# Test using a variable declared without initial assignment (will be None) -- expecting TypeError
print(f"\nTest case: Using a variable declared without initial assignment (expecting error)")
input_expression_4 = "my_weight + 10.0;"
print(f"\nInput: {input_expression_4}")
try:
    output_4_expr, my_variables = run_interpreter(input_expression_4, my_variables)
    if output_4_expr is not None: print(f"Result: {output_4_expr}")
except TypeError as e:
    print(f"Caught expected error: {e}")


# Test using a variable that is not defined -- expecting NameError
print(f"\nTest case: Using an undefined variable (expecting error)")
input_expression_5 = "undefined_variable + 1;"
print(f"\nInput: {input_expression_5}")
try:
    output_5_expr, my_variables = run_interpreter(input_expression_5, my_variables)
    if output_5_expr is not None: print(f"Result: {output_5_expr}")
except NameError as e:
    print(f"Caught expected error: {e}")

# Test case for kennel definition
input_kennel = "kennel MyDoggo { treats default_treats = 5; weight default_weight; }"
print(f"\nInput: {input_kennel}")
_, my_variables = run_interpreter(input_kennel, my_variables)

# Test case for import statement
input_import = "import mymodule;"
print(f"\nInput: {input_import}")
_, my_variables = run_interpreter(input_import, my_variables)

# Test case 1: Basic if statement (true condition)
input_if_1 = "if (10 > 5) { treats result = 1; };"
print(f"\nInput: {input_if_1}")
_, my_variables = run_interpreter(input_if_1, my_variables)
print(f"Result of if (10 > 5): result = {my_variables.get('result')}")

# Test case 2: Basic if statement (false condition)
input_if_2 = "if (10 < 5) { treats result = 2; };"
print(f"\nInput: {input_if_2}")
_, my_variables = run_interpreter(input_if_2, my_variables)
print(f"Result of if (10 < 5): result = {my_variables.get('result')}") # Should still be 1 or previous value

# Test case 3: if-sniff_else statement (if true)
my_variables['my_treats'] = 10.0 # Reset value for consistent testing
input_if_else_1 = "if (my_treats > 5) { treats status = 100; } sniff_else { treats status = 0; };"
print(f"\nInput: {input_if_else_1}")
_, my_variables = run_interpreter(input_if_else_1, my_variables)
print(f"Result of if-else (my_treats > 5): status = {my_variables.get('status')}")

# Test case 4: if-sniff_else statement (else true)
my_variables['my_treats'] = 3.0 # Change value to test else branch
input_if_else_2 = "if (my_treats > 5) { treats status = 100; } sniff_else { treats status = 0; };"
print(f"\nInput: {input_if_else_2}")
_, my_variables = run_interpreter(input_if_else_2, my_variables)
print(f"Result of if-else (my_treats = 3): status = {my_variables.get('status')}")

# Test case 5: if-sniff_elif-sniff_else statement (if true)
my_variables['my_treats'] = 10.0 # Reset value
input_if_elif_else_1 = "if (my_treats == 10) { treats branch = 1; } sniff_elif (my_treats > 5) { treats branch = 2; } sniff_else { treats branch = 3; };"
print(f"\nInput: {input_if_elif_else_1}")
_, my_variables = run_interpreter(input_if_elif_else_1, my_variables)
print(f"Result of if-elif-else (my_treats == 10): branch = {my_variables.get('branch')}")

# Test case 6: if-sniff_elif-sniff_else statement (elif true)
my_variables['my_treats'] = 7.0 # Change value
input_if_elif_else_2 = "if (my_treats == 10) { treats branch = 1; } sniff_elif (my_treats > 5) { treats branch = 2; } sniff_else { treats branch = 3; };"
print(f"\nInput: {input_if_elif_else_2}")
_, my_variables = run_interpreter(input_if_elif_else_2, my_variables)
print(f"Result of if-elif-else (my_treats = 7): branch = {my_variables.get('branch')}")

# Test case 7: if-sniff_elif-sniff_else statement (else true)
my_variables['my_treats'] = 2.0 # Change value
input_if_elif_else_3 = "if (my_treats == 10) { treats branch = 1; } sniff_elif (my_treats > 5) { treats branch = 2; } sniff_else { treats branch = 3; };"
print(f"\nInput: {input_if_elif_else_3}")
_, my_variables = run_interpreter(input_if_elif_else_3, my_variables)
print(f"Result of if-elif-else (my_treats = 2): branch = {my_variables.get('branch')}")

# Test case for while loop
my_variables['counter'] = 0.0
input_while = "while (counter < 3) { counter = counter + 1; treats loop_val = counter * 10; };"
print(f"\nInput: {input_while}")
_, my_variables = run_interpreter(input_while, my_variables)
print(f"Result of while loop: counter = {my_variables.get('counter')}, loop_val = {my_variables.get('loop_val')}")

# Test case for for loop (with declaration, condition, and increment)
my_variables['sum_for'] = 0.0
input_for_full = "for (treats i = 0; i < 3; i = i + 1) { sum_for = sum_for + i; };"
print(f"\nInput: {input_for_full}")
_, my_variables = run_interpreter(input_for_full, my_variables)
print(f"Result of for loop (full): sum_for = {my_variables.get('sum_for')}")

# Unit test for while loop with multiple statements and complex condition
my_variables['x'] = 0.0
my_variables['y'] = 0.0
input_while_test = "while (x < 5 and y < 3) { x = x + 1; y = y + 1; };"
print(f"\nUnit Test Input: {input_while_test}")
_, my_variables = run_interpreter(input_while_test, my_variables)
assert my_variables['x'] == 3.0, f"Expected x to be 3.0, but got {my_variables['x']}"
assert my_variables['y'] == 3.0, f"Expected y to be 3.0, but got {my_variables['y']}"
print(f"Unit Test for while loop passed: x = {my_variables['x']}, y = {my_variables['y']}")

# Unit test for for loop with no initialization and expression for increment
my_variables['j'] = 5.0
my_variables['product_for'] = 1.0
input_for_test_2 = "for (; j > 2; j = j - 1) { product_for = product_for * j; };"
print(f"\nUnit Test Input: {input_for_test_2}")
_, my_variables = run_interpreter(input_for_test_2, my_variables)
# Loop should run for j=5, 4, 3. product_for = 1 * 5 * 4 * 3 = 60
assert my_variables['j'] == 2.0, f"Expected j to be 2.0, but got {my_variables['j']}"
assert my_variables['product_for'] == 60.0, f"Expected product_for to be 60.0, but got {my_variables['product_for']}"
print(f"Unit Test for for loop (no init, expr increment) passed: j = {my_variables['j']}, product_for = {my_variables['product_for']}")

# Test case for comments
input_comments = "# This is a single-line comment\ntreats comment_test = 1; /* This is a multi-line comment */\n/* Another multi-line\n * comment */ my_treats = comment_test + 1; # End of line comment";
print(f"\nInput: {input_comments}")
_, my_variables = run_interpreter(input_comments, my_variables)
print(f"Result of comment test (comment_test): {my_variables.get('comment_test')}")
print(f"Result of comment test (my_treats): {my_variables.get('my_treats')}")

# New Test cases for try-catch-finally and growl
print(f"\n--- New Tests: Try-Catch-Finally and Growl ---")

# Test 1: Basic try-finally (no exception)
my_variables['try_test_val'] = 0.0
input_try_finally_1 = "try { treats try_test_val = 10; } sniff_finally { try_test_val = try_test_val + 1; };"
print(f"\nInput: {input_try_finally_1}")
_, my_variables = run_interpreter(input_try_finally_1, my_variables)
print(f"Result of try-finally (no exception): try_test_val = {my_variables.get('try_test_val')}")
assert my_variables['try_test_val'] == 11.0

# Test 2: try-growl-finally (growl in try block)
my_variables['try_test_val'] = 0.0
my_variables['final_val'] = 0.0
input_try_growl_finally_1 = "try { treats try_test_val = 10; growl Exception(\"Error in try\"); } sniff_finally { final_val = 99; };"
print(f"\nInput: {input_try_growl_finally_1}")
try:
    _, my_variables = run_interpreter(input_try_growl_finally_1, my_variables)
except Exception as e:
    print(f"Caught expected error outside interpreter: {e}")
print(f"Result of try-growl-finally (growl in try): try_test_val = {my_variables.get('try_test_val')}, final_val = {my_variables.get('final_val')}")
assert my_variables['try_test_val'] == 10.0
assert my_variables['final_val'] == 99.0

# Test 3: try-growl-catch-finally (growl in try, caught by sniff_catch)
my_variables['try_test_val'] = 0.0
my_variables['catch_val'] = 0.0
my_variables['final_val'] = 0.0
input_try_catch_growl_finally_1 = "try { growl Exception(\"Caught error\"); } sniff_catch (Exception e) { treats catch_val = 1; } sniff_finally { treats final_val = 2; };"
print(f"\nInput: {input_try_catch_growl_finally_1}")
_, my_variables = run_interpreter(input_try_catch_growl_finally_1, my_variables)
print(f"Result of try-growl-catch-finally: catch_val = {my_variables.get('catch_val')}, final_val = {my_variables.get('final_val')}, caught_exception_message = {my_variables.get('e')}")
assert my_variables['catch_val'] == 1.0
assert my_variables['final_val'] == 2.0
assert str(my_variables['e']) == "Caught error"

# Test 4: emptybowl literal
input_emptybowl = "weight my_null_var = emptybowl;"
print(f"\nInput: {input_emptybowl}")
_, my_variables = run_interpreter(input_emptybowl, my_variables)
print(f"Result of emptybowl test: my_null_var = {my_variables.get('my_null_var')}")
assert my_variables['my_null_var'] is None


# New unit test for diary type with default None value (without emptybowl)
input_diary_declaration_test = "diary new_diary_entry;"
print(f"\nUnit Test (diary type default): {input_diary_declaration_test}")
_, my_variables = run_interpreter(input_diary_declaration_test, my_variables)
assert my_variables.get('new_diary_entry') is None, f"Expected 'new_diary_entry' to be None, but got {my_variables.get('new_diary_entry')}"
print(f"Unit test passed: 'new_diary_entry' is {my_variables.get('new_diary_entry')}")


# Unit test for peek method (formerly fetchfile)
print(f"\n--- New Test: peek method ---")
dummy_filename = "test_diary.txt"
dummy_content = "Hello from the Doggo diary!"

# Create a dummy file
with open(dummy_filename, 'w') as f:
    f.write(dummy_content)

# Declare a diary variable
input_diary_declaration = "diary my_diary;"
print(f"\nInput: {input_diary_declaration}")
_, my_variables = run_interpreter(input_diary_declaration, my_variables)
assert my_variables.get('my_diary') is None, f"Expected 'my_diary' to be None initially, but got {my_variables.get('my_diary')}"

# Call peek method
input_peek_call = f"my_diary.peek(\"{dummy_filename}\");"
print(f"\nInput: {input_peek_call}")
_, my_variables = run_interpreter(input_peek_call, my_variables)

# Assert content is loaded
assert my_variables.get('my_diary') == dummy_content, f"Expected 'my_diary' to contain '{dummy_content}', but got '{my_variables.get('my_diary')}'"
print(f"Unit test for peek passed: my_diary contains '{my_variables.get('my_diary')}'")

# Unit test for nap method
print(f"\n--- New Test: nap method ---")
input_nap_call = "my_diary.nap();"
print(f"\nInput: {input_nap_call}")
_, my_variables = run_interpreter(input_nap_call, my_variables)

# Assert content is cleared
assert my_variables.get('my_diary') == "", f"Expected 'my_diary' to be empty string after nap, but got '{my_variables.get('my_diary')}'"
print(f"Unit test for nap passed: my_diary is '{my_variables.get('my_diary')}'")


# Unit test for leave_scent function (write mode)
print(f"\n--- New Test: leave_scent function (write mode) ---")
write_filename_w = "test_write_w.txt"
write_content_w = "First line written.\n"
input_leave_scent_w_func = f"leave_scent(\"w\", \"{write_filename_w}\", \"{write_content_w}\");"
print(f"\nInput: {input_leave_scent_w_func}")
_, my_variables = run_interpreter(input_leave_scent_w_func, my_variables)

with open(write_filename_w, 'r') as f:
    read_content_w = f.read()
assert read_content_w == write_content_w, f"Expected file '{write_filename_w}' to contain '{write_content_w}', but got '{read_content_w}'"
print(f"Unit test for leave_scent function (write mode) passed: '{write_filename_w}' content is '{read_content_w}'")

# Unit test for leave_scent function (append mode)
print(f"\n--- New Test: leave_scent function (append mode) ---")
write_content_a = "Second line appended.\n"
input_leave_scent_a_func = f"leave_scent(\"a\", \"{write_filename_w}\", \"{write_content_a}\");"
print(f"\nInput: {input_leave_scent_a_func}")
_, my_variables = run_interpreter(input_leave_scent_a_func, my_variables)

expected_content_a = write_content_w + write_content_a
with open(write_filename_w, 'r') as f:
    read_content_a = f.read()
assert read_content_a == expected_content_a, f"Expected file '{write_filename_w}' to contain '{expected_content_a}', but got '{read_content_a}'"
print(f"Unit test for leave_scent function (append mode) passed: '{write_filename_w}' content is '{read_content_a}'")

# Unit test for leave_scent function (invalid mode)
print(f"\n--- New Test: leave_scent function (invalid mode) ---")
input_leave_scent_invalid_func = f"leave_scent(\"x\", \"invalid.txt\", \"some content\");"
print(f"\nInput: {input_leave_scent_invalid_func}")
try:
    _, my_variables = run_interpreter(input_leave_scent_invalid_func, my_variables)
except ValueError as e:
    print(f"Caught expected error: {e}")
    assert "Invalid file operation mode" in str(e)
print(f"Unit test for leave_scent function (invalid mode) passed.")

# Unit test for bury_bone (successful deletion)
print(f"\n--- New Test: bury_bone function (successful deletion) ---")
delete_filename = "file_to_delete.txt"
# Create the file to be deleted
with open(delete_filename, 'w') as f:
    f.write("This file will be deleted.")
assert os.path.exists(delete_filename), f"Pre-condition failed: file '{delete_filename}' should exist."

input_bury_bone_func = f"bury_bone(\"{delete_filename}\");"
print(f"\nInput: {input_bury_bone_func}")
_, my_variables = run_interpreter(input_bury_bone_func, my_variables)

assert not os.path.exists(delete_filename), f"Post-condition failed: file '{delete_filename}' should have been deleted."
print(f"Unit test for bury_bone (successful deletion) passed: '{delete_filename}' no longer exists.")

# Unit test for bury_bone (deleting non-existent file)
print(f"\n--- New Test: bury_bone function (non-existent file) ---")
non_existent_file = "non_existent_file.txt"
assert not os.path.exists(non_existent_file), f"Pre-condition failed: file '{non_existent_file}' should not exist."

input_bury_bone_non_existent = f"bury_bone(\"{non_existent_file}\");"
print(f"\nInput: {input_bury_bone_non_existent}")
_, my_variables = run_interpreter(input_bury_bone_non_existent, my_variables)
# In this case, we expect a DEBUG message but no error if it's handled gracefully.
# If you change the interpreter to raise an error for non-existent files, you'd wrap this in a try-except.
print(f"Unit test for bury_bone (non-existent file) passed (handled gracefully).")

# New unit tests for file extension validation
print(f"\n--- New Test: File Extension Validation ---")

# Test peek with invalid extension
invalid_file_ext_peek = "my_diary.peek(\"invalid.jpg\");"
print(f"\nInput: {invalid_file_ext_peek}")
try:
    _, my_variables = run_interpreter(invalid_file_ext_peek, my_variables)
except ValueError as e:
    print(f"Caught expected error: {e}")
    assert "Unsupported file extension" in str(e)
print(f"Unit test for peek with invalid extension passed.")

# Test leave_scent with invalid extension
invalid_file_ext_leave_scent = "leave_scent(\"w\", \"invalid.pdf\", \"some content\");"
print(f"\nInput: {invalid_file_ext_leave_scent}")
try:
    _, my_variables = run_interpreter(invalid_file_ext_leave_scent, my_variables) # Corrected variable name
except ValueError as e:
    print(f"Caught expected error: {e}")
    assert "Unsupported file extension" in str(e)
print(f"Unit test for leave_scent with invalid extension passed.")

# Test bury_bone with invalid extension
invalid_file_ext_bury_bone = "bury_bone(\"invalid.exe\");"
print(f"\nInput: {invalid_file_ext_bury_bone}")
try:
    _, my_variables = run_interpreter(invalid_file_ext_bury_bone, my_variables)
except ValueError as e:
    print(f"Caught expected error: {e}")
    assert "Unsupported file extension" in str(e)
print(f"Unit test for bury_bone with invalid extension passed.")

# Unit test for beg (input)
print(f"\n--- New Test: beg (input) ---")
input_beg_no_prompt = "beg(my_input_var);"
print(f"\nInput: {input_beg_no_prompt}")
# For testing in Colab, input() will prompt the user directly.
# If you run this in a non-interactive setup, it might raise an EOFError or block.
# For now, we'll let it run and expect a default/simulated input if interaction is not possible.
# If a human agent is running this, they would manually provide input.
_, my_variables = run_interpreter(input_beg_no_prompt, my_variables)
print(f"Value after beg (no prompt): {my_variables.get('my_input_var')}")
# Depending on interaction, this might be 'fallback_input' or actual user input.
assert 'my_input_var' in my_variables

input_beg_with_prompt = "beg(another_input_var, \"Enter a number: \");"
print(f"\nInput: {input_beg_with_prompt}")
_, my_variables = run_interpreter(input_beg_with_prompt, my_variables)
print(f"Value after beg (with prompt): {my_variables.get('another_input_var')}")
assert 'another_input_var' in my_variables

# Unit test for woof (print)
print(f"\n--- New Test: woof (print) ---")
my_variables['print_test_var'] = 123.45
input_woof_literal = "woof(\"Hello, Doggo!\");"
print(f"\nInput: {input_woof_literal}")
# This test will visually check the output in the standard_output
_, my_variables = run_interpreter(input_woof_literal, my_variables)

input_woof_var = "woof(print_test_var);"
print(f"\nInput: {input_woof_var}")
_, my_variables = run_interpreter(input_woof_var, my_variables)

input_woof_expression = "woof(my_treats + 10);"
print(f"\nInput: {input_woof_expression}")
_, my_variables = run_interpreter(input_woof_expression, my_variables)

# --- New Math Function Tests ---
print(f"\n--- New Tests: Math Functions ---")

# Test sqrt
input_sqrt = "treats s = sqrt(25);"
print(f"\nInput: {input_sqrt}")
_, my_variables = run_interpreter(input_sqrt, my_variables)
print(f"Result of sqrt(25): s = {my_variables.get('s')}")
assert my_variables['s'] == 5.0

# Test pawsize (abs)
input_pawsize = "treats p = pawsize(-10);"
print(f"\nInput: {input_pawsize}")
_, my_variables = run_interpreter(input_pawsize, my_variables)
print(f"Result of pawsize(-10): p = {my_variables.get('p')}")
assert my_variables['p'] == 10.0

# Test smallest_pup (min)
input_smallest_pup = "treats min_val = smallest_pup(10, 5, 20);"
print(f"\nInput: {input_smallest_pup}")
_, my_variables = run_interpreter(input_smallest_pup, my_variables)
print(f"Result of smallest_pup(10, 5, 20): min_val = {my_variables.get('min_val')}")
assert my_variables['min_val'] == 5.0

# Test biggest_pup (max)
input_biggest_pup = "treats max_val = biggest_pup(10, 5, 20);"
print(f"\nInput: {input_biggest_pup}")
_, my_variables = run_interpreter(input_biggest_pup, my_variables)
print(f"Result of biggest_pup(10, 5, 20): max_val = {my_variables.get('max_val')}")
assert my_variables['max_val'] == 20.0

# Test alpha_power (pow)
input_alpha_power = "treats power_val = alpha_power(2, 3);"
print(f"\nInput: {input_alpha_power}")
_, my_variables = run_interpreter(input_alpha_power, my_variables)
print(f"Result of alpha_power(2, 3): power_val = {my_variables.get('power_val')}")
assert my_variables['power_val'] == 8.0


print(f"\nAll interpreter test cases ran successfully!")
print(f"Final variables after all test cases: {my_variables}")


Input: treats my_treats = 10;
-- Parse Tree --
(start (statement (declaration (type_ treats) (identifier my_treats) = (expr (atom 10))) ;) <EOF>)
------------------

Input: weight my_weight;
-- Parse Tree --
(start (statement (declaration (type_ weight) (identifier my_weight)) ;) <EOF>)
------------------

Input: my_treats + 5;
-- Parse Tree --
(start (statement (expr (expr (atom (identifier my_treats))) + (expr (atom 5))) ;) <EOF>)
------------------
Result: 15.0

Input: leash dog_names = ["Max", "Rex"];
-- Parse Tree --
(start (statement (declaration (type_ leash) (identifier dog_names) = (expr (list_literal [ (element "Max") , (element "Rex") ]))) ;) <EOF>)
------------------
DEBUG: ListLiteral text child: ["Max","Rex"]
DEBUG: List literal fallback parsed items: ['Max', 'Rex']
Variables after leash declaration: {'my_treats': 10.0, 'my_weight': None, 'dog_names': ['Max', 'Rex']}

Input: dog_names.leashon("Buddy");
-- Parse Tree --
(start (statement (list_method_statement (identifier

## Clean up

In [179]:
# Clean up generated ANTLR Python files do not run
!rm -f Doggo.g4 DoggoLexer.py DoggoParser.py DoggoListener.py DoggoVisitor.py Doggo.tokens DoggoLexer.tokens Doggo.interp DoggoLexer.interp \
    test_diary.txt test_write_w.txt

print("Cleaned up generated ANTLR Python files.")

Cleaned up generated ANTLR Python files.
